# distill-lab runner

runs any distill-lab command on a kaggle gpu. edit `COMMANDS` below, then **Save Version → Save & Run All** so it keeps running after you close the tab.

one-time setup in the notebook sidebar:
- **Accelerator**: GPU T4 x2 (one T4 is enough for now)
- **Internet**: on (needs a phone-verified kaggle account)
- **Add-ons → Secrets**: `HF_TOKEN` (write token from huggingface.co/settings/tokens), and `WANDB_API_KEY` if you use `--wandb`

In [ ]:
REPO = "https://github.com/avichal-08/distill-lab.git"
BRANCH = "main"

COMMANDS = [
    "python -m distill_lab.eval.run_eval --model Qwen/Qwen3-0.6B-Base --name b0",
]

# hf dataset repo to copy outputs/ into when done, e.g. "your-hf-name/distill-lab-results". None to skip
UPLOAD_TO = None

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    for key in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[key] = secrets.get_secret(key)
            print(key, "loaded")
        except Exception:
            print(key, "not set")
except ImportError:
    print("not running on kaggle, using the environment as is")

# without a key, --wandb would sit waiting for a login prompt
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_MODE"] = "disabled"

In [ ]:
!rm -rf /kaggle/working/distill-lab
!git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/distill-lab
%cd /kaggle/working/distill-lab
!git log --oneline -1
!pip install -q -e ".[gpu]"

In [ ]:
!python -m distill_lab.data.prepare --out cache/data

In [ ]:
for cmd in COMMANDS:
    print(">>", cmd, flush=True)
    get_ipython().system(cmd)
    if _exit_code != 0:
        raise RuntimeError(f"failed with exit code {_exit_code}: {cmd}")

In [ ]:
if UPLOAD_TO:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(UPLOAD_TO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path="outputs", path_in_repo="outputs", repo_id=UPLOAD_TO, repo_type="dataset")
    print(f"uploaded to https://huggingface.co/datasets/{UPLOAD_TO}")
else:
    print("outputs stay in /kaggle/working/distill-lab/outputs (kept with the saved version)")

### if something breaks

- **vllm crashes while starting up**: add `--eager` to the command (turns off cuda graphs, a bit slower)
- **out of memory**: lower `eval.max_new_tokens`, or run one command per session
- **sessions stop after 12 hours** and the weekly gpu quota is 30 hours, so split long jobs